<a href="https://colab.research.google.com/github/vestrada-data/Analisis_Comercial/blob/main/Analisis_de_Caso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.


In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders_original = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog_original = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing_original = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [ ]:

#hacer una copia del dataset
orders = orders_original.copy()
catalog = catalog_original.copy()
marketing = marketing_original.copy()
# explorar datasets
print("=== Revisamos datos ===")
orders.info()
orders.head()


=== Revisamos datos ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


- Dataset con 25100 registros
- Se identifican columnas con valores faltantes y datos con tipo de dato por cambiar.

In [ ]:
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


- Canal con información faltante y fechas con tipo de dato objeto

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas
---

In [ ]:
# Convertir las columnas fecha_hora_pedido y fecha a  tipo fecha
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha']=pd.to_datetime(marketing['fecha'], errors='coerce')
print("\n=== Convertimos fechas en 'tipo fecha' y validamos===")
#validar cambios
print(orders['fecha_hora_pedido'].dtypes)
print(marketing['fecha'].dtypes)


=== Convertimos fechas en 'tipo fecha' y validamos===
datetime64[ns]
datetime64[ns]


In [ ]:
print(f"Revisamos registros con fechas erróneas después 2026: {(orders['fecha_hora_pedido'].dt.year > 2026).sum()}")

Revisamos registros con fechas erróneas después 2026: 0


In [ ]:
print("\n=== Revisamos variables numéricas sin negativos o ceros inválidos ===")

# Investigar el comportamiento general
print(f"Valores de descuento {orders['monto_descuento'].value_counts()}")

# Función para revisar variables numéricas (sin negativos o ceros inválidos)
def revision_col_numerica(df, columnas):
  for col in columnas:
    print(f"\n--- Análisis de la columna: {col} ---")
    print(f"Valores nulos reales (NaN): {df[col].isnull().sum()}")
    print(f"Valores en cero (0.0): {(df[col] == 0).sum()}")
    print(f"Valores negativos (< 0): {(df[col] < 0).sum()}")
    print(f"Valor sentilen -999:  {df[col].isin([-999]).sum()}")
col_num=['cantidad','precio_unitario', 'monto_descuento', 'monto_total']
col_num2=['costo_unitario']
col_num3=['gasto']

#Aplicar función

revision_col_numerica(orders,col_num)
revision_col_numerica(catalog,col_num2)
revision_col_numerica(marketing,col_num3)

# Guardamos la lista de columnas que quieres ver
columnas_deseadas = ['id_pedido', 'pais', 'cantidad', 'precio_unitario', 'monto_total']

# Filtramos: filas donde 'cantidad' es nula, y seleccionamos las columnas
resultado = orders.loc[orders['cantidad'].isnull(), columnas_deseadas]

print(resultado)


=== Revisamos variables numéricas sin negativos o ceros inválidos ===
Valores de descuento 0.0     12551
10.0     5068
5.0      4940
15.0     2491
Name: monto_descuento, dtype: int64

--- Análisis de la columna: cantidad ---
Valores nulos reales (NaN): 50
Valores en cero (0.0): 0
Valores negativos (< 0): 4
Valor sentilen -999:  0

--- Análisis de la columna: precio_unitario ---
Valores nulos reales (NaN): 50
Valores en cero (0.0): 0
Valores negativos (< 0): 0
Valor sentilen -999:  0

--- Análisis de la columna: monto_descuento ---
Valores nulos reales (NaN): 50
Valores en cero (0.0): 12551
Valores negativos (< 0): 0
Valor sentilen -999:  0

--- Análisis de la columna: monto_total ---
Valores nulos reales (NaN): 0
Valores en cero (0.0): 0
Valores negativos (< 0): 4
Valor sentilen -999:  0

--- Análisis de la columna: costo_unitario ---
Valores nulos reales (NaN): 0
Valores en cero (0.0): 0
Valores negativos (< 0): 0
Valor sentilen -999:  0

--- Análisis de la columna: gasto ---
Valores

In [ ]:
#porcentaje de nulos en tabla Orders
print(f'Porcentajes de nulos en Dataset orders:')
print(round((orders.isna().sum() / 25100 * 100),2))

# Se observaron cantidades muy grandes, por lo que se verifica el contexto
print("\nCompras con cantidades de productos mayores a 10000:")
columnas_analisis = ['id_pedido', 'id_usuario', 'dispositivo', 'pais', 'cantidad', 'monto_total']
orders[orders['cantidad'] >= 10000][columnas_analisis]

Porcentajes de nulos en Dataset orders:
id_pedido             0.00
id_usuario            0.00
fecha_hora_pedido     0.00
pais                  1.20
dispositivo           0.08
fuente_referencia     0.12
nombre_producto       0.12
categoria_producto    0.32
cantidad              0.20
precio_unitario       0.20
monto_descuento       0.20
monto_total           0.00
dtype: float64

Compras con cantidades de productos mayores a 10000:


,id_pedido,id_usuario,dispositivo,pais,cantidad,monto_total
3521,order_3521,user_5812,mobile,Mexico,10000.0,431400.0
3522,order_3522,user_3575,desktop,Argentina,10000.0,2805500.0
3586,order_3586,user_3380,mobile,Mexico,10000.0,4903500.0
3643,order_3643,user_4440,desktop,Colombia,10000.0,2381500.0
3656,order_3656,user_884,mobile,Argentina,20000.0,5953200.0
3668,order_3668,user_7270,mobile,Mexico,20000.0,6966200.0
3689,order_3689,user_6566,desktop,mexico,10000.0,876900.0
3722,order_3722,user_4723,mobile,Argentina,20000.0,8840200.0
3726,order_3726,user_2536,desktop,Colombia,20000.0,5817000.0
3748,order_3748,user_7096,desktop,Mexico,10000.0,3369300.0


<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

Se detectan diez pedidos con cantidades de 10.000 o 20.000 unidades, se mantienen ya que tienen un impacto importante en el monto final.
</div>

In [ ]:

# Ver si los negativos corresponden a algún estatus de orden específico
print("Compras con cantidades negativas")
print(orders[orders['cantidad'] < 0])
print()
# Ver solo las columnas importantes de las filas con cantidad NaN para identificar algún patrón
print(orders[orders['cantidad'].isnull()][columnas_analisis].head())
print()

Compras con cantidades negativas
     id_pedido id_usuario fecha_hora_pedido pais dispositivo  \
266  order_266  user_7011        2025-03-13  NaN     desktop   
267  order_267  user_1087        2025-05-07  NaN     desktop   
268  order_268    user_84        2025-02-19  NaN     desktop   
269  order_269  user_3323        2025-05-25  NaN     desktop   

    fuente_referencia  nombre_producto categoria_producto  cantidad  \
266       paid_search  Phone-Pro-128GB        Electronica      -2.0   
267            social  Phone-Pro-128GB        Electronica      -1.0   
268           organic  Phone-Pro-128GB        Electronica      -1.0   
269       paid_search  Phone-Pro-128GB        Electronica      -1.0   

     precio_unitario  monto_descuento  monto_total  
266           101.31             10.0      -192.62  
267            43.50              5.0       -38.50  
268           497.65              5.0      -492.65  
269           423.53              0.0      -423.53  

   id_pedido id_usuario 

In [ ]:

# Verificar cuántos duplicados hay por el identificador único del pedido
duplicados = orders[orders.duplicated(subset=['id_pedido'], keep=False)]
print('Registros duplicados')
orders['id_pedido'].value_counts().head(10)

Registros duplicados


order_6083     2
order_14695    2
order_4533     2
order_4326     2
order_12374    2
order_14267    2
order_14566    2
order_11623    2
order_4907     2
order_8326     2
Name: id_pedido, dtype: int64

In [ ]:
#Duplicados en Catalog y Marketing
print(f"Registros duplicados en Catalog: {catalog['nombre_producto'].duplicated().sum()}")
print(f"Registros duplicados en Marketing: {marketing[['id_campaña','fecha','gasto']].duplicated().sum()}")

Registros duplicados en Catalog: 0
Registros duplicados en Marketing: 0


In [ ]:
#Revision de datos duplicados
display(orders[orders['id_pedido'] == 'order_3323'])
duplicados = orders[orders.duplicated(subset=['id_pedido'], keep='first')]
print(f"Registos duplicados por id_pedido: {len(duplicados)} equivalentes al {round(((len(duplicados)/25100)*100),2)}% del total")



,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3323,order_3323,user_1763,2025-05-19,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,42.08,0.0,42.08
25006,order_3323,user_1763,2025-05-19,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,42.08,0.0,42.08


Registos duplicados por id_pedido: 100 equivalentes al 0.4% del total


In [ ]:
# Revisar MCAR y MAR
pais_nulo=orders['pais'].isnull().sum()
print(f"Registros con país nulo: {pais_nulo} equivalentes al {((pais_nulo/25100)*100).round(2)}% del total")
paises_con_nulos = orders[orders['pais'].isna()]
paises_con_nulos.head()

Registros con país nulo: 300 equivalentes al 1.2% del total


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
124,order_124,user_6671,2025-04-12,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,298.44,5.0,293.44
125,order_125,user_5263,2025-05-30,NaN,mobile,organic,Tablet-Standard-64GB,Electronica,1.0,133.13,0.0,133.13
126,order_126,user_4952,2025-04-19,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,2.0,177.23,0.0,354.45
127,order_127,user_2619,2025-06-26,NaN,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,329.53,10.0,319.53
128,order_128,user_2444,2025-05-06,NaN,desktop,social,Vacuum-Pro-Black,Hogar,2.0,218.43,0.0,436.86


In [ ]:

#Verificación agrupando país con dispositivo, producto y fuente de referencia
print("Verificación MAR  para pais")
print()
display(orders['pais'].isna().groupby(orders['nombre_producto']).mean().sort_values(ascending=False))
display(orders['pais'].isna().groupby(orders['dispositivo']).mean().sort_values(ascending=False))
display(orders['pais'].isna().groupby(orders['fuente_referencia']).mean().sort_values(ascending=False))
display(orders['pais'].isna().groupby(orders['fecha_hora_pedido']).mean().sort_values(ascending=False))
print(f"Monto total registros con PAÍS NULO: ${orders[orders['pais'].isna()]['monto_total'].sum():.2f}")


Verificación MAR  para pais



nombre_producto
Blender-XL-Red          0.014064
Phone-Pro-128GB         0.013091
Sneakers-Urban-42       0.012500
Vacuum-Pro-Black        0.012384
Tablet-Standard-64GB    0.011871
Jacket-Winter-M         0.010258
Laptop-Gaming-16GB      0.008948
Name: pais, dtype: float64

dispositivo
mobile     0.012499
desktop    0.011443
Name: pais, dtype: float64

fuente_referencia
paid_search    0.013954
social         0.012814
organic        0.009125
Name: pais, dtype: float64

fecha_hora_pedido
2025-02-19    0.038760
2025-06-27    0.036496
2025-06-16    0.035714
2025-04-15    0.034722
2025-06-09    0.034483
                ...   
2025-05-28    0.000000
2025-04-22    0.000000
2025-01-09    0.000000
2025-03-01    0.000000
2025-04-01    0.000000
Name: pais, Length: 181, dtype: float64

Monto total registros con PAÍS NULO: $110840.75


In [ ]:
# Modificamos el Dataset
# Eliminar duplicados de pedidos conservando la primera ocurrencia
orders.drop_duplicates(subset=['id_pedido'], keep='first', inplace=True)

#Validamos cambios
print(f"Verificamos cambios:")
print(orders.shape)
print(f"DUPLICADOS: {orders.duplicated().sum()}")
print()
#Validamos el monto total de registros con dispositivo Nulo
nulos_cantidad   = orders[orders['cantidad'].isna()]
print("=== Revisamos registros con dispositivo NULO ===")
print(f"Cantidad            : {len(nulos_cantidad)}")
print(f"Monto total suma    : ${nulos_cantidad['monto_total'].sum():,.2f}")

Verificamos cambios:
(25000, 12)
DUPLICADOS: 0

=== Revisamos registros con dispositivo NULO ===
Cantidad            : 50
Monto total suma    : $21,628.00


In [ ]:
#Eliminación de registros
print("\n=== Eliminamos los 50 registros con cantidad  y precio unitario nulo ya que solo representan el 0.20% y el monto total que aportan no es significativo")
#eliminar los 50 registros con NaN en cantidad (0.2% del dataset) que también eliminan los de precio_unitario
orders.dropna(subset=['cantidad'], inplace=True)

print("\n=== Rellenamos Monto descuento NaN como cero===")
#rellenamos NaN en monto_descuento con ceros
orders['monto_descuento'].fillna(0, inplace=True)

orders.reset_index(drop=True)
orders.shape
#Verificamos cambios
print(f"Verificamos cambios:")
print(f"Registros con MONTO DESCUENTO Nulo: {orders['monto_descuento'].isnull().sum()}")


#Cambiamos los 300 registros sin país a 'Desconocido' para análisis por país
print('\n=== Cambiamos países con NaN a Desconocido===')
orders['pais'] = orders['pais'].fillna('Desconocido')

#Verificamos país
print(orders['pais'].value_counts(dropna=False))
print()

# Filtrar para dejar solo las cantidades mayores a 0 (eliminar 4 registros negativos)
orders = orders[orders['cantidad'] > 0]

# Verificar que ya no existan los 50 NaN
print(f"\nRegistros con CANTIDAD nula: {orders['cantidad'].isnull().sum()}")

# Verificar que ya no existan números negativos
print('\nValidamos que no existen números negativos:')
print(orders['cantidad'].value_counts(dropna=False))



=== Eliminamos los 50 registros con cantidad  y precio unitario nulo ya que solo representan el 0.20% y el monto total que aportan no es significativo

=== Rellenamos Monto descuento NaN como cero===
Verificamos cambios:
Registros con MONTO DESCUENTO Nulo: 0

=== Cambiamos países con NaN a Desconocido===
Colombia       7467
Mexico         7465
Argentina      7239
mexico          862
colombia        822
argentina       795
Desconocido     300
Name: pais, dtype: int64


Registros con CANTIDAD nula: 0

Validamos que no existen números negativos:
2.0        12591
1.0        12345
10000.0        6
20000.0        4
Name: cantidad, dtype: int64


In [ ]:
#Revisamos variables categóricas
print("=== Revisamos Variables Categorías ===")
print(orders.shape)

#Registros sin nombre_prodcuto
print(f"Total de registros sin nombre de producto:{orders['nombre_producto'].isnull().sum()}")
print(f"Total de registros sin Canal de marketing:{marketing['canal'].isnull().sum()}")
print(marketing[marketing['canal'].isnull()])

# Rellenamos canal nulo (NaN) con el correspondiente
def extraer_canal(id_campania):
    # Si el ID es nulo, devolvemos None
    if pd.isna(id_campania):
        return None

    # Separamos el texto por los guiones bajos
    partes = str(id_campania).split('_')

    # Condición: si hay 3 o más palabras, une las dos primeras
    if len(partes) >= 3:
        return "_".join(partes[:2])
    # Condición: si hay 2 palabras (o menos), se queda con la primera
    else:
        return partes[0]

# Creamos la serie con los canales extraídos
canales_extraidos = marketing['id_campaña'].apply(extraer_canal)

print('\nLlenamos canal con el valor correspondiente')
# Rellenamos los NaN de la columna 'canal'
marketing['canal'] = marketing['canal'].fillna(canales_extraidos)
print(f"Validamos resultado: Total de registros sin Canal de marketing:{marketing['canal'].isnull().sum()}")

#eliminamos registros con nombre_producto como NaN ya que es llave secundaria
print("\n=== Eliminamos registros  con nombre_producto como NaNy validamos ===")
orders = orders.dropna(subset=['nombre_producto'])

#reiniciamos índice
orders.reset_index(drop=True)
print(orders.shape)

print("\n===Normalizamos columnas categoricas y verificamos===")
lista1=['pais', 'dispositivo','fuente_referencia']
lista2=['categoria_producto','proveedor']
lista3=['id_campaña','canal']

def normalizar_categoricos (df, columnas):
    for col in columnas:
         df[col] = df[col].str.strip().str.title()
         print(df[col].value_counts())
         print()
normalizar_categoricos(orders, lista1)
normalizar_categoricos(catalog, lista2)
normalizar_categoricos(marketing, lista3)

# Estandarizar en ambas tablas (modifica los originales)
orders['nombre_producto'] = orders['nombre_producto'].str.strip().str.replace('Gb', 'GB', regex=False).str.replace('Xl', 'XL', regex=False)
# Verificar
orders.head()

=== Revisamos Variables Categorías ===
(24946, 12)
Total de registros sin nombre de producto:30
Total de registros sin Canal de marketing:101
         fecha       pais             id_campaña canal    gasto
98  2025-01-11  Argentina       social_Argentina   NaN   849.70
99  2025-01-12     Mexico         organic_Mexico   NaN  2033.56
100 2025-01-12     Mexico     paid_search_Mexico   NaN  1260.65
101 2025-01-12     Mexico          social_Mexico   NaN  1660.90
102 2025-01-12   Colombia       organic_Colombia   NaN  1819.27
..         ...        ...                    ...   ...      ...
194 2025-01-22   Colombia        social_Colombia   NaN   772.46
195 2025-01-22  Argentina      organic_Argentina   NaN  1562.61
196 2025-01-22  Argentina  paid_search_Argentina   NaN   644.34
197 2025-01-22  Argentina       social_Argentina   NaN  2544.66
198 2025-01-23     Mexico         organic_Mexico   NaN  2977.84

[101 rows x 5 columns]

Llenamos canal con el valor correspondiente
Validamos resultado: 

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,Desktop,Social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
display(marketing.head())
catalog.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,Organic_Mexico,Organic,2446.25
1,2025-01-01,Mexico,Paid_Search_Mexico,Paid_Search,2704.34
2,2025-01-01,Mexico,Social_Mexico,Social,2045.01
3,2025-01-01,Colombia,Organic_Colombia,Organic,2597.21
4,2025-01-01,Colombia,Paid_Search_Colombia,Paid_Search,1771.40


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena And Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers Llc
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr And Finley"


In [ ]:
# Ver estadísticas de monto_total de los 20 registros nulos vs el resto
nulos_dispositivo   = orders[orders['dispositivo'].isna()]
print("=== Revisamos registros con dispositivo NULO ===")
print(f"Cantidad            : {len(nulos_dispositivo)}")
print(f"Monto total suma    : ${nulos_dispositivo['monto_total'].sum():,.2f}")

=== Revisamos registros con dispositivo NULO ===
Cantidad            : 20
Monto total suma    : $6,355.75


In [ ]:
#eliminacion de regitros con dispositivo nulo
orders = orders.dropna(subset=['dispositivo'])
orders.reset_index(drop=True)
print(f"Filas restantes: {len(orders)}")
orders['cantidad'] = orders['cantidad'].astype(int)

#verificamos nulos
print("\n=== Verificamos nulos ===")
print(f"Filas         : {len(orders)}")
print(f"NaN por columna:\n{orders.isna().sum()}")
print(orders[['precio_unitario', 'cantidad', 'monto_descuento', 'monto_total']].head())

Filas restantes: 24896

=== Verificamos nulos ===
Filas         : 24896
NaN por columna:
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64
   precio_unitario  cantidad  monto_descuento  monto_total
0           332.69         2              0.0       665.37
1           176.86         1              5.0       171.86
2           102.99         2             10.0       195.99
3           257.87         1             15.0       242.87
4           336.28         1              0.0       336.28


In [ ]:
#Verificamos consistencia de montos
# Convertir descuento a decimal real
orders['descuento_decimal'] = orders['monto_descuento'] / 100

# Monto total correcto
orders['monto_total_correcto'] = (orders['precio_unitario'] * orders['cantidad']
                                  * (1 - orders['descuento_decimal'])).round(2)

# Verificar
print(orders[['precio_unitario', 'cantidad', 'monto_descuento',
              'descuento_decimal', 'monto_total',
              'monto_total_correcto']].head(10))


   precio_unitario  cantidad  monto_descuento  descuento_decimal  monto_total  \
0           332.69         2              0.0               0.00       665.37   
1           176.86         1              5.0               0.05       171.86   
2           102.99         2             10.0               0.10       195.99   
3           257.87         1             15.0               0.15       242.87   
4           336.28         1              0.0               0.00       336.28   
5           179.34         1              0.0               0.00       179.34   
6           163.59         2              0.0               0.00       327.19   
7           373.68         2              0.0               0.00       747.35   
8           477.27         2              5.0               0.05       949.54   
9           339.30         1              5.0               0.05       334.30   

   monto_total_correcto  
0                665.38  
1                168.02  
2                185.38  
3   

In [ ]:

# Ver distribución de diferencias
orders['diferencia']  = (orders['monto_total'] - orders['monto_total_correcto']).abs().round(2)# Separar por tipo de diferencia
redondeo  = orders[(orders['diferencia'] > 0.01) &
                        (orders['diferencia'] <= 1)]
inconsistentes = orders[orders['diferencia'] > 1]

print("=== Verificamos consistencia de montos ===")
print(f"Registros correctos      : {len(orders[orders['diferencia'] == 0])}")
print(f"Diferencia por redondeo  : {len(orders[orders['diferencia'] <= 0.01])}")
print(f"Inconsistencias reales   : {len(inconsistentes)}")

# Ver si hay patrón por producto
print("\n=== Inconsistencias por producto ===")
print(inconsistentes['nombre_producto'].value_counts())

# Ver si hay patrón por país
print("\n=== Inconsistencias por país ===")
print(inconsistentes['pais'].value_counts())


=== Verificamos consistencia de montos ===
Registros correctos      : 9364
Diferencia por redondeo  : 12479
Inconsistencias reales   : 11898

=== Inconsistencias por producto ===
Blender-XL-Red          2014
Jacket-Winter-M         2013
Vacuum-Pro-Black        1989
Sneakers-Urban-42       1960
Laptop-Gaming-16GB      1355
Phone-Pro-128GB         1285
Tablet-Standard-64GB    1282
Name: nombre_producto, dtype: int64

=== Inconsistencias por país ===
Mexico         4046
Colombia       3950
Argentina      3760
Desconocido     142
Name: pais, dtype: int64


In [ ]:
# inconsistencias en 11,898 registros (47.8% del dataset), recalculamos monto_total
print("Inconsistencias en 11,898 registros (47.8% del dataset), utilizaremos  monto_total_correcto para calculos posteriores")

# Ver valores exactos de costo_unitario en catalog
print(catalog[['nombre_producto', 'costo_unitario']])
print(f"\nTipo de dato: {catalog['costo_unitario'].dtype}")
print(f"Promedio    : {catalog['costo_unitario'].mean():,.2f}")
print(f"Mínimo      : {catalog['costo_unitario'].min():,.2f}")
print(f"Máximo      : {catalog['costo_unitario'].max():,.2f}")

#Verificamos precios unitarios mayores a costos unitarios (fuga de dinero)
print("\n===Verificamos precios unitarios mayores a costos unitarios (fuga de dinero)===")
#Unir tablas
tabla_temp =pd.merge(orders,catalog, on=['nombre_producto'], how='left')
condicion_fuga = tabla_temp['precio_unitario'] < tabla_temp['costo_unitario']
# Contar cuántos registros cumplen la condición
total_registros_fuga = condicion_fuga.sum()
print(f"Número de registros con precio menor al costo: {total_registros_fuga}")


Inconsistencias en 11,898 registros (47.8% del dataset), utilizaremos  monto_total_correcto para calculos posteriores
        nombre_producto  costo_unitario
0    Laptop-Gaming-16GB          280.68
1       Phone-Pro-128GB           10.12
2  Tablet-Standard-64GB           25.21
3        Blender-XL-Red          176.64
4      Vacuum-Pro-Black           16.60
5     Sneakers-Urban-42           17.21
6       Jacket-Winter-M          189.31

Tipo de dato: float64
Promedio    : 102.25
Mínimo      : 10.12
Máximo      : 280.68

===Verificamos precios unitarios mayores a costos unitarios (fuga de dinero)===
Número de registros con precio menor al costo: 4380


**Documenta el hallazgo en el EDA**

**Fechas**
- Las columnas fecha_hora_pedido y fecha se convierten a tipo fecha con pd.to_datetime()
- Verificamos que no haya fechas futuras (mayor a 2026)
  
**Columnas numéricas**
- Existen 50 registros con NaN en cantidad y precio unitario (0.2% del dataset). Ya que no es posible verificar si el monto total es correcto y al representar un porcentaje muy bajo del dataset, son eliminados del análisis.
- Se identificaron 100 registros duplicados exactos, equivalentes a 0.40% del conjunto de datos. Al tratarse de observaciones completamente idénticas, se eliminaron utilizando drop_duplicates() para evitar el doble conteo en los análisis y visualizaciones.
- Existen 4 registros con 'cantidad' negativa que son eliminados mediante fillna().
- Hay 50 registros cuyo Monto descuento  es nulo y lo cambiamos a cero para cálculos posteriores.
- 10 registros con montos mayores a 10000 se mantinenen debido al impacto en el monto final.
  
**Variables categóricas**  
- La variable país presenta 300 valores faltantes (1.2% del total). Al no haber un patrón claro asociado a fecha, dispositivo, referencia o producto y debido al bajo porcentaje de datos faltantes, los registros se conservaron y se etiquetaron como "Desconocido" utilizando fillna() para identificarlos en los análisis descriptivos por país.
-Se identificaron 30 registros con 'nombre_producto' nulo. Al ser una llave secundaria y no permitir valores nulos, se eliminaron del dataset para su análisis posterior en Power BI.
-Rellenamos 101 canales que tenian valores nulos con el correspondiente, tomado del id_campaña.

**Validacion de montos y fuga de dinero**
- Inconsistencias en montos en 11,898 registros (47.8% del dataset). Se calculan los montos correctos utilizando una nueva columna monto_total_correcto para cálculos posteriores, dejando la información anterior en la columna original monto_total.
- Se detectan 4380 registros en donde el  precio unitario del producto es  menor al costo unitario, generando una pérdida importante de dinero.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:

# exportar datasets
orders.to_csv('orders_clean.csv', index=False, decimal='.', encoding='utf-8-sig')
catalog.to_csv('catalog_clean.csv', index=False, decimal='.', encoding='utf-8-sig')
marketing.to_csv('marketing_clean.csv', index=False, decimal='.', encoding='utf-8-sig')


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# Cálculos de rentabilidad
#Ingreso Total
KPI_ingreso_total=orders['monto_total_correcto'].sum()

# Unir datasets
tabla_temporal =pd.merge(orders,catalog, on=['nombre_producto'], how='inner')

#Cálculo del costo
costo_por_fila = tabla_temporal['cantidad'] * tabla_temporal['costo_unitario']
#Sumamos el resultado para obtener el KPI
KPI_costo_total = costo_por_fila.fillna(0).sum()
KPI_costo_total = round(KPI_costo_total, 2)

#Inversión en marketing
KPI_gasto_campana= marketing['gasto'].sum()

#Calculo de Profit
tabla_temporal['descuento_monto'] = (tabla_temporal['precio_unitario'] * tabla_temporal['cantidad']
                              * (tabla_temporal['monto_descuento'] / 100)).round(2)
tabla_temporal['profit']= (tabla_temporal['monto_total_correcto']
                    - (tabla_temporal['cantidad'] * tabla_temporal['costo_unitario'])).round(2)
 # Suma total del profit
KPI_profit = tabla_temporal['profit'].sum()

# Suma únicamente de las pérdidas (valores negativos)
perdida = tabla_temporal.loc[tabla_temporal['profit'] < 0, 'profit'].sum()


print('RENTABILIDAD DEL NEGOCIO')
print(f"\n Ingreso total:, ${KPI_ingreso_total:,.2f}")
print(f"\n Costo total:, ${KPI_costo_total:,.2f}")
print(f"\n Gasto en Marketing:, ${KPI_gasto_campana:,.2f}")
print(f"\n Ganancia bruta:, ${KPI_profit:,.2f}")
print(f"\n Ganancia :, ${(KPI_profit -KPI_gasto_campana) :,.2f}")
print(f"\n Perdida:, ${perdida:,.2f}")


RENTABILIDAD DEL NEGOCIO

 Ingreso total:, $51,624,990.41

 Costo total:, $43,122,388.54

 Gasto en Marketing:, $2,871,843.53

 Ganancia:, $8,502,601.87

 Perdida:, $-5,415,872.37


In [ ]:
#Ticket promedio
# Contar cuántos usuarios únicos hicieron pedido
total_pedidos = orders['id_pedido'].nunique()
ticket_promedio = (KPI_ingreso_total / total_pedidos).round(2)

#Cantidad promedio de productos por orden
avg_cantidad=orders['cantidad'].mean()

#Producto más vendido
#Agrupa por producto y sumar las cantidades
ventas_por_producto = orders.groupby('nombre_producto')['cantidad'].sum()
# Encuentra el que tiene la cantidad máxima
mas_vendido = ventas_por_producto.idxmax()

#Gasto en  marketing por canal
KPI_gasto_canal = marketing.groupby('canal')['gasto'].sum()

print('COMPORTAMIENTO DE VENTAS')
print(f"\nTicket promedio:, ${ticket_promedio:,.2f}")
print(f"Cantidad promedio de productos por orden:{avg_cantidad:.2f}")
print(f"Producto más vendido:, {mas_vendido}")
print(f"\nGasto en Marketing por Canal de Marketing")
print(KPI_gasto_canal.map('${:,.2f}'.format))

COMPORTAMIENTO DE VENTAS

Ticket promedio:, $2,073.63
Cantidad promedio de productos por orden:7.13
Producto más vendido:, Laptop-Gaming-16GB

Gasto en Marketing por Canal de Marketing
canal
Organic        $972,650.96
Paid_Search    $922,374.20
Social         $976,818.37
Name: gasto, dtype: object


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
WITH cte_first_visit AS (
    SELECT
        id_usuario,
        MIN(timestamp_evento) AS ts_first_visit
    FROM events
    WHERE nombre_evento = 'first_visit'
    GROUP BY id_usuario
),

cte_select_item AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_select_item
    FROM events e
    INNER JOIN cte_first_visit fv
        ON e.id_usuario = fv.id_usuario
    WHERE e.nombre_evento = 'select_item'
      AND e.timestamp_evento > fv.ts_first_visit
    GROUP BY e.id_usuario
),

cte_add_to_cart AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_add_to_cart
    FROM events e
    INNER JOIN cte_select_item si
        ON e.id_usuario = si.id_usuario
    WHERE e.nombre_evento = 'add_to_cart'
      AND e.timestamp_evento > si.ts_select_item
    GROUP BY e.id_usuario
),

cte_begin_checkout AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_begin_checkout
    FROM events e
    INNER JOIN cte_add_to_cart ac
        ON e.id_usuario = ac.id_usuario
    WHERE e.nombre_evento = 'begin_checkout'
      AND e.timestamp_evento > ac.ts_add_to_cart
    GROUP BY e.id_usuario
),

cte_add_payment AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_add_payment
    FROM events e
    INNER JOIN cte_begin_checkout bc
        ON e.id_usuario = bc.id_usuario
    WHERE e.nombre_evento = 'add_payment_info'
      AND e.timestamp_evento > bc.ts_begin_checkout
    GROUP BY e.id_usuario
),

cte_purchase AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_purchase
    FROM events e
    INNER JOIN cte_add_payment ap
        ON e.id_usuario = ap.id_usuario
    WHERE e.nombre_evento = 'purchase'
      AND e.timestamp_evento > ap.ts_add_payment
    GROUP BY e.id_usuario
)

SELECT
    (SELECT COUNT(*) FROM cte_first_visit)    AS first_visit_users,
    (SELECT COUNT(*) FROM cte_select_item)    AS select_item_users,
    (SELECT COUNT(*) FROM cte_add_to_cart)    AS add_to_cart_users,
    (SELECT COUNT(*) FROM cte_begin_checkout) AS begin_checkout_users,
    (SELECT COUNT(*) FROM cte_add_payment)    AS add_payment_info_users,
    (SELECT COUNT(*) FROM cte_purchase)       AS purchase_users;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,first_visit_users,add_to_cart_users,add_payment_info_users,purchase_users
0,7796,7634,6250,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS (
    SELECT
        id_usuario,
        MIN(timestamp_evento) AS ts_first_visit
    FROM events
    WHERE nombre_evento = 'first_visit'
    GROUP BY id_usuario
),

cte_select_item AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_select_item
    FROM events e
    INNER JOIN cte_first_visit fv
        ON e.id_usuario = fv.id_usuario
    WHERE e.nombre_evento = 'select_item'
      AND e.timestamp_evento > fv.ts_first_visit
    GROUP BY e.id_usuario
),

cte_cart AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_add_to_cart
    FROM events e
    INNER JOIN cte_select_item si
        ON e.id_usuario = si.id_usuario
    WHERE e.nombre_evento = 'add_to_cart'
      AND e.timestamp_evento > si.ts_select_item
    GROUP BY e.id_usuario
),

cte_begin_checkout AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_begin_checkout
    FROM events e
    INNER JOIN cte_cart c
        ON e.id_usuario = c.id_usuario
    WHERE e.nombre_evento = 'begin_checkout'
      AND e.timestamp_evento > c.ts_add_to_cart
    GROUP BY e.id_usuario
),

cte_addpayment AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_add_payment
    FROM events e
    INNER JOIN cte_begin_checkout bc
        ON e.id_usuario = bc.id_usuario
    WHERE e.nombre_evento = 'add_payment_info'
      AND e.timestamp_evento > bc.ts_begin_checkout
    GROUP BY e.id_usuario
),

cte_purchase AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS ts_purchase
    FROM events e
    INNER JOIN cte_addpayment ap
        ON e.id_usuario = ap.id_usuario
    WHERE e.nombre_evento = 'purchase'
      AND e.timestamp_evento > ap.ts_add_payment
    GROUP BY e.id_usuario
)

SELECT

ROUND(
    (SELECT COUNT(*) FROM cte_select_item) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_first_visit),0),
2) AS conversion_first_to_select_item,

ROUND(
    (SELECT COUNT(*) FROM cte_cart) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_select_item),0),
2) AS conversion_select_item_to_cart,

ROUND(
    (SELECT COUNT(*) FROM cte_begin_checkout) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_cart),0),
2) AS conversion_cart_to_begin_checkout,

ROUND(
    (SELECT COUNT(*) FROM cte_addpayment) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_begin_checkout),0),
2) AS conversion_begin_checkout_to_add_info,

ROUND(
    (SELECT COUNT(*) FROM cte_purchase) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_addpayment),0),
2) AS conversion_add_info_to_purchase,

ROUND(
    (SELECT COUNT(*) FROM cte_purchase) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM cte_first_visit),0),
2) AS conversion_final;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion



,conversion_first_to_cart,conversion_cart_to_add_info,conversion_add_info_to_payment,conversion_final
0,97.92,81.87,99.84,80.04



**ANÁLISIS DEL FUNNEL**

- **First Visit → Add to Cart (97.92%):** La gran mayoría de los usuarios que visitan el sitio agregan al menos un producto al carrito, lo que indica un alto nivel de interés inicial.
  
- **Add to Cart → Add Payment Info (81.87%)**: En esta etapa se observa la mayor disminución de usuarios. Aproximadamente el 18.13% abandona el proceso antes de proporcionar la información de pago, por lo que este es el principal punto de fuga del embudo y una oportunidad de mejora.

- **Add Payment Info → Purchase (99.84%):** Una vez que los usuarios ingresan su información de pago, solo el 0.16% no finaliza la compra, lo que sugiere que el proceso de pago funciona de manera eficiente.

- **Conversión final (80.04%):** El 80.04% de los usuarios que iniciaron el recorrido completaron una compra. En términos generales, el embudo presenta un buen desempeño, aunque existe margen de mejora en la transición entre el carrito de compras y el ingreso de la información de pago.


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users


# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
display(users.head(3))
users.info()



,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_usuario      8000 non-null   object
 1   fecha_registro  8000 non-null   object
 2   país            8000 non-null   object
 3   dispositivo     8000 non-null   object
 4   tipo_plan       8000 non-null   object
dtypes: object(5)
memory usage: 312.6+ KB


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
display(user_activity.head(3))
user_activity.info()

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             32000 non-null  object
 1   fecha_actividad        32000 non-null  object
 2   dias_despues_registro  32000 non-null  int64 
 3   activo                 32000 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1000.1+ KB


In [ ]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    -- Cohorte por mes de registro
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes
    FROM users
),

retencion AS (
    -- Unimos users y user_activity y clasificamos  por semana
    SELECT
        c.id_usuario,
        c.cohorte_mes,

        -- Retención por semana usando dias_despues_registro
        MAX(
    CASE
        WHEN FLOOR(ua.dias_despues_registro / 7) >= 1
             AND ua.activo = 1
        THEN 1
        ELSE 0
    END
) AS retenido_w1,

MAX(
    CASE
        WHEN FLOOR(ua.dias_despues_registro / 7) >= 2
             AND ua.activo = 1
        THEN 1
        ELSE 0
    END
) AS retenido_w2,

MAX(
    CASE
        WHEN FLOOR(ua.dias_despues_registro / 7) >= 3
             AND ua.activo = 1
        THEN 1
        ELSE 0
    END
) AS retenido_w3
    FROM cohortes c
    JOIN user_activity ua ON c.id_usuario = ua.id_usuario
    GROUP BY c.id_usuario, c.cohorte_mes
),

agregado AS (
    -- sumamos por cohorte
    SELECT
        cohorte_mes,
        COUNT(id_usuario)       AS clientes_iniciales,
        SUM(retenido_w1)        AS retenido_w1,
        SUM(retenido_w2)        AS retenido_w2,
        SUM(retenido_w3)        AS retenido_w3
    FROM retencion
    GROUP BY cohorte_mes
)

-- Paso 4: Calcular porcentajes
SELECT
    cohorte_mes,
    clientes_iniciales,

    retenido_w1,
    retenido_w2,
    retenido_w3,

    ROUND(retenido_w1 * 100.0 / clientes_iniciales, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / clientes_iniciales, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / clientes_iniciales, 2) AS semana_3

FROM agregado
ORDER BY cohorte_mes;
'''
# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,cohorte_mes,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La modificación en la UI del checkout no tiene efecto sobre la tasa de conversión. La tasa de conversión del grupo de tratamiento es igual a la del grupo de control.
   - **H₁ (Hipótesis alternativa):** La modificación en la UI del checkout sí impacta la tasa de conversión. Existe una diferencia significativa entre ambos grupos.
   
**Test estadístico:** Se aplicó un Z-test de dos proporciones dado que la métrica de interés es binaria (conversión: sí/no), y con una muestra grande la distribución se comporta de forma normal."

**Nivel de significancia alpha:** "Se establece un nivel de significancia de α = 0.05, lo que significa que aceptamos un 5% de probabilidad de concluir que la nueva UI tiene efecto cuando en realidad no lo tiene. Es el estándar de la industria para pruebas A/B porque balancea el riesgo de tomar una decisión incorrecta con la sensibilidad suficiente para detectar diferencias reales en el comportamiento del usuario."

In [ ]:
# importamos librerias
from statsmodels.stats.proportion import proportions_ztest
# revisamos dataset
experiment= pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
# Contar número de éxitos y número total de registros
conversiones = experiment.groupby('variante')['convirtio'].sum()
totales      = experiment.groupby('variante')['convirtio'].count()

# Pasar los valores a formato lista — control primero como base
exitos        = [conversiones['control'], conversiones['tratamiento']]
observaciones = [totales['control'],      totales['tratamiento']]

# Aplicar prueba y visualizar resultados
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")
# Interpretar resultados
alpha = 0.05  # umbral de significancia
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de que la nueva UI impacta la tasa de conversión.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de impacto de la nueva UI.")


Estadístico z: -0.8132782986429474
Valor p: 0.41605851639119995
No rechazamos la hipótesis nula: no hay evidencia suficiente de impacto de la nueva UI.


In [ ]:
tasa_A= exitos[0] / observaciones[0]
tasa_B= exitos[0] / observaciones[0]
print(f"Tasa de conversión página A: {tasa_A:.2%}")
print(f"Tasa de conversión página B: {tasa_B:.2%}")

if tasa_A > tasa_B:
    print(f"\nLa página A tiene una mayor tasa de conversión ({tasa_A - tasa_B:.2%}).")
elif tasa_B > tasa_A:
    print(f"\nLa página B tiene una mayor tasa de conversión ({tasa_B - tasa_A:.2%})")
else:
    print("\nAmbas páginas tienen la misma tasa de conversión.")

Tasa de conversión página A: 15.69%
Tasa de conversión página B: 15.69%

Ambas páginas tienen la misma tasa de conversión.


---

**CONCLUSIÓN E INTERPRETACIÓN Conclusión (test estadístico)**

**Decisión:**  
No rechazamos la hipótesis nula: no hay evidencia suficiente de impacto de la nueva UI.

**Interpretación de negocio:**  
La tasa de conversión fue prácticamente idéntica entre ambos grupos — 15.69% tanto en control como en tratamiento. El Z-test confirma que esta diferencia mínima no es estadísticamente significativa (Z = -0.81, p = 0.42), por lo que no rechazamos la hipótesis nula. La modificación en la UI del checkout no tuvo ningún impacto sobre la decisión de compra del usuario.

---

**📊RESUMEN EJECUTIVO**

**1. Objetivos del Análisis**
El presente análisis tiene como propósito evaluar el rendimiento comercial global y los márgenes de ganancia de la operación, así como determinar la frecuencia de compra y el valor generado por usuario en los mercados de Argentina, México y Colombia. A través de este diagnóstico, se busca identificar ineficiencias operativas y formular recomendaciones estratégicas para proteger la rentabilidad del negocio.

**2. Hallazgos Clave**
- El ingreso total acumulado del periodo fue de `$51.62` millones, generado principalmente por el segmento 'Electrónica' a lo largo de los tres mercados operativos: Argentina, México y Colombia.  
- El producto Laptop-Gaming-16GB representa el 88% de la facturación total, pero registra una fuga de `$5.03` millones debido a transacciones donde el precio de venta es menor al costo unitario, concentrándose el impacto en México.  
- Argentina lidera la rentabilidad del negocio al generar `$5.35 millones` millones en utilidades con una facturación de `$20.62` millones (un margen de conversión del 26%). En contraste, México presenta una severa ineficiencia operativa: a pesar de facturar casi lo mismo (`$19.64 millones`), solo retiene $1.52 millones en utilidades (un margen de apenas 7.7%).

 **3. Métricas Principales**
>
> * **Ingreso Total:** \$51.62 millones
> * **Ganancia Total:** \$5.63 millones
> * **Fuga de Margen Total:** <span style="color:red">**\$5.38 millones**</span>
> * **Gasto de Marketing:** \$2.87 millones
> * **Ticket Promedio:** \$2.07 mil
> * **Productos Promedio por Orden:** 7.13 unidades

 **3. Insights accionables**
- La operación de RappiPlus presenta un riesgo de concentración crítico. Al depender casi en su totalidad de la categoría 'Electrónica' (y específicamente de la venta de hardware de alta gama), el flujo del negocio es altamente vulnerable a factores externos incontrolables. El incremento en los aranceles de importación en América Latina o una desaceleración en el consumo de bienes durables impactaría de forma inmediata y severa las finanzas de la empresa. Las categorías de 'Hogar' y 'Moda' se encuentran actualmente subutilizadas, desaprovechando su potencial como amortiguadores de ingresos.
- El éxito comercial del producto Laptop-Gaming-16GB está subsidiando una pérdida invisible; el sistema actual permite aplicar descuentos ciegos que destruyen el margen neto.
- La baja conversión de México no es un problema de demanda o de mercado (pues el volumen de ventas es casi idéntico al de Argentina), sino el síntoma directo de la distorsión de precios de la Laptop-Gaming-16GB. El mercado mexicano está absorbiendo el volumen de transacciones en pérdida, erosionando el margen del país.

 **4. Recomendaciones Estratégicas**
- Ofrecer incentivos o cupones de descuento en las categorías de 'Hogar' o 'Moda' al momento de adquirir un artículo de 'Electrónica' de alto valor (como la laptop).
- Implementar una regla de validación tecnológica en la base de datos (bloqueo automático) para que ningún precio de venta pueda quedar por debajo del costo unitario cargado.
- Replicar las políticas de control de precios y subsidios aplicadas en la operación de Argentina en el mercado mexicano. Auditar las campañas de descuento locales de México de manera inmediata para frenar la erosión de valor.

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:
--

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores
---
2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico evolución mensual de revenue o profit
- Gráfico YTD
- Gráfico revenue y profit por producto o categoría
---
 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.